# LAB 1 - wprowadzenie do tematyki LLM
Termin odesłania notatnika - 16.03. Proszę odsyłać notebooki na maila marton@agh.edu.pl (tytuł maila według schematu: APIS_2026_lab1_imie_nazwisko)


## Opisz pojęcia związane z LLM.
Wytłumacz czym jest:
- Token
- Kontekst modelu
- Embedding
- Predykcja kolejnego tokenu


In [ ]:
## Miejsce na twoją odpowiedź

`Token` - niewielki element tekstu (najczęściej znak, ale również słowo, sylaba) na podstawie którego operuje model językowy. Przed wytrenowaniem modelu tekst rozdziela się na tokeny (tokenizacja), aby model uznał te same znaki/wyrazy/sylaby jako to samo zjawisko (uznał je za identyczny token).

`Kontekst modelu` - informacje jakie model trzyma w pamięci podczas generowania odpowiedzi na prompt. W praktyce oznacza to maksymalną liczbę tokenów, jaką model może przetwarzać w pojedynczym przewidywaniu. Składa się z chociażby: treści promptu, historii rozmowy, tekstu już wygenerowanego przez model.

`Embedding` - jest wektorowa reprezentacja tekstu w przestrzeni matematycznej. Niemożliwe jest przeprowadzanie operacji matematycznych na tekście, stąd konwersja do wektorów jest kluczowa.

`Predykcja kolejnego tokenu` - Mechanizm predykcji tokenu polega na przewidywaniu następnego znaku w danej sekwencji (tekście). Na początku model dostaje ciąg tokenów, a następnie oblicza za pomocą prawdopodobieństwa, jaki znak powinien wystąpić jako następny. Ten proces można powtarzać wielokrotnie, aby uzyskiwać dłuższe sekwencje.

Opisz schemat działania modeli LLM od otrzymania promptu od użytkownika do wygenerowania odpowiedzi

In [ ]:
## Miejsce na twoją odpowiedź

*Zakładam, że model został wytrenowany, wagi są znane, model jest gotowy do użycia.
1. Otrzymanie promptu  
STO LAT STO LAT

2. Tokenizacja promptu
Załóżmy wyrazową: T1, T2, T1, T2    
T1 - STO    
T2 - LAT    

3. Embedding tokenów    
T1 - [0.43, 0.21, 0.11, ...]    
T2 - [0.33, 0.31, 0.26, ...]    

4. Transformer - badanie zależności między tokenami, ich podobieństwa względem siebie   
Przykładowo: STO może mu się skojarzyć z liczbą, wiekiem, procentami, itp...

5. Predykcja kolejnego tokenu na podstawie prawdopodobieństwa w sposób iteracyjny   
Przykładowo: Model bierze pod uwagę kilka ostatnich tokenów i zauważa, że gdy dwukrotnie wypisane jest STO LAT, to kolejnym tokenem powinno być NIECH   
W kolejnej iteracji dodaje słowo ŻYJE 
I tak dalej aż uzna, że najbardziej prawdopodbne jest zakończyć przewidywanie lub ma założoną maksymalna ilosć tokenów, które może wygenerować


## Ćwiczenie na tokenizację
Opdal poniższy kod do dwóch metod tokenizowania tekstu, opisz czym różnią się wyniki dla otrzymanych metod. Opisz na czym może polegać różnica w wyborze odpowiedniego tokenizera dla odpowiedniego modelu. Podaj przykłady metod tokenizacji.

In [ ]:
%pip install tiktoken transformers sentencepiece --quiet

In [5]:
text = "sto lat sto lat"

In [6]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(text)
decoded_tokens = [encoding.decode([t]) for t in tokens]

print("Liczba tokenów:", len(tokens))
print("Tokeny:")
print(decoded_tokens)

Liczba tokenów: 4
Tokeny:
['sto', ' lat', ' sto', ' lat']


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokens = tokenizer.tokenize(text)
ids = tokenizer.convert_tokens_to_ids(tokens)

print("Liczba tokenów:", len(tokens))
print("Tokeny:")
print(tokens)

Liczba tokenów: 8
Tokeny:
['st', '##o', 'la', '##t', 'st', '##o', 'la', '##t']


In [ ]:
## Miejsce na twoją odpowiedź

Pierwsza metoda używa tokenizera z biblioteki OpenAI (tiktoken, encoding cl100k_base). Tokenizer ten dzieli tekst na większe fragmenty (tzw. subword chunks). Druga metoda pochodzi z modeli dostępnych w bibliotece Hugging Face (bert-base-uncased). Tokenizer ten dzieli słowa na mniejsze części. Symbol ## oznacza, że token jest kontynuacją poprzedniego fragmentu słowa, stąd większa liczba tokenów właśnie w tej metodzie

Dobór tokenizera generalnie mocno zależy od modelu, ponieważ model jest już trenowany razem z konkretnym tokenizerem. Zatem użycie innego tokenizeraa może prowadzić do błędnego rozumienia tekstu przez model, gorszej jakości generacji, niepoprawnych embeddingów, itp.

Najprostsze tokenizery to te dzielące tekst na wyrazy (oddzielone spacjami) oraz na znaki. Inne, bardziej zaawansowane to chociażby Unigram Language Model Tokenization (wybiera najlepszy zestaw tokenów z probabilistycznego słownika), Byte Pair Encoding (łączy najczęstsze pary znaków w większe tokeny)

## Transformery

Wyjaśnij pojęcia:
- attention
- self-attention
- wagi uwagi (attention weights)
- multi-head attention

In [ ]:
## Miejsce na twoją odpowiedź

`Attention` - to mechanizm pozwalający modelowi skupić się na najważniejszych elementach danych wejściowych. Przykładowo w zdaniu: "Alicja szła sobie spokojnie i nagle upadła", model skupia się bardziej na słowach "Alicja", "szła", "upadła" i im przypisuje wyższe wagi.

`Self-attention` - to szczególny przypadek attention, w którym elementy sekwencji zwracają uwagę na inne elementy tej samej sekwencji. Przykładowo: "Alicja szła do Kasi i się przewróciła". Analizując słowo "przewróciła" model musi ustalić o którą dziewczynę chodzi, więc ywkorzystuje mechanizm self-attention, aby spojrzeć na wszystkie inne słowa zdania i zdecydować, które są istotne. Self-attention umożliwia modelowi uchwycenie zależności długodystansowych.

`attention weights` - to liczby/współczynniki określające jak bardzo jeden element powinien zwracać uwagę na inny. Przykładowo w zdaniu "Alicja szła sobie spokojnie i nagle upadła", słowo "szła" będzie miało wysoką wagę w kontekście "Alicja", ale niską w kontekście "nagle".

`multi-head attention` - oznacza, że mechanizm attention jest wykonywany kilka razy równolegle.

Dlaczego transformery są lepsze do rozwiązań LLM od klasycznych modeli do przetwarzania sekwencji (LSTM/RNN)?

In [ ]:
## Miejsce na twoją odpowiedź

Transformery są szybsze w treningu (można zrównoleglać proces uczenia modelu), lepiej łapią kontekst w danej sekwencji (inaczej mówiąc lepiej rozumieją kontekst) i łatwo skalują się do dużych modeli językowych.

## Typy systemów opartych o LLM
Wytłumacz czym są i czym różnią się względem siebie
- Chatbot
- RAG
- Agent AI

In [12]:
## Miejsce na twoją odpowiedź

Chatbot to najprostszy system wykorzystujący LLM do prowadzenia rozmowy z użytkownikiem. RAG to architektura, w której LLM przed wygenerowaniem odpowiedzi wyszukuje informacje w jakiejś zewnętrznej bazie wiedzy. AI agent to bardziej złożony system, w którym LLM podejmuje działania, korzystając z różnych narzędzi.  
Główna różnica między tymi trzeba systemami jest w źródle z jakiego korzystają oraz w działaniu. Chatboty korzystaja z wytrenowanego modelu jako źródła wiedzy, a ich działanie to po prostu rozmowa, konwersacja z użytkownikiem. RAG wykorzystuje dodatkowo rózne przekazane przez użytkownika dokumenty oraz na ich podstawie prowadzi konwersację. Agenci dodatkowo wykorzystują różnego rodzaju narzędzia jako źródło, a dodatkowo potrafi planować i wykonywać zadania (np. wywołać jakiś kod).

## Parametry generacji
Omów do czego służą parametry:
- temperature
- top_p
- top_k
- max_tokens

In [13]:
## Miejsce na twoją odpowiedź

`temperature` - kontroluje losowość generacji. Im mniejsza wartość, tym odpowiedzi są bardziej przewidywalne i mniej losowe. Im wyższa wartość, tym model jest bardziej kreatywny, choć czasami mniej spójny, bardziej chaotyczny

`top-p` - ogranicza wybór tokenów do mniejszego zbioru tokenów za pomocą prawdopodbieństwa. Innymi słowy eliminuje bardzo mało prawdopodobne tokeny.

`top-k` - ogranicza wybór do najbardziej prawdopodobnych tokenów.

`max_tokens` - określa maksymalną liczbę tokenów, które model może wygenerować w odpowiedzi. W skrócie - żeby odpowiedź nie była za długa.

## Sposoby korzystania z modeli
Opisz dwa główne sposoby korzystania z modeli: 
- Pobierając model lokalnie
- Korzystając z API.

Przestaw wady, zalety, możliwości i potencjalne zastosowania obu podejść.

In [15]:
## Miejsce na twoją odpowiedź

`Pobieranie modelu lokalnie` - Polega na pobraniu wag modelu i uruchomieniu go na własnym sprzęcie.

Zalety: 
- pełna kontrola nad modelem
- brak wysyłania danych do zewnętrznych usług 
- możliwość modyfikacji, fine-tuningu modelu
- brak opłat za zapytania do API

Wady:
- często wysokie wymagania sprzętowe (mocne GPU, dużo RAM)

Typowe zastosowania:
- projekty badawcze i eksperymenty ML
- aplikacje wymagające pełnej prywatności danych
- systemy działające offline

`Korzystanie z API` - Polega na wysyłaniu zapytań do modelu hostowanego przez dostawcę, typu OpenAI 

Zalety:
- prostota w obsudze (w sumie to wystarczy wywołanie API)
- dostęp do bardzo dużych wytrenowanych i aktualnych modeli
- brak potrzeby posiadania własnej infrastruktury
- skalowalność i szybkie wdrożenie

Wady:
- koszty za użycie
- zależność od dostawcy
- dane są wysyłane do zewnętrznej usługi
- mniejsza możliwość modyfikacji modelu

Typowe zastosowania:
- chatboty i asystenci AI
- aplikacje webowe z funkcjami AI

Przeanalizuj i opisz koszty związane z używaniem modeli przy użyciu API. Porównaj różne pakiety dla różnych wersji GPT oraz dla HuggingFace Inference API.

In [1]:
## Miejsce na twoją odpowiedź

Spytałem o to samego zainteresowanego i zwrócił mi takie koszty (w zależności od modelu):   

GPT-4o mini	- input ok. $0.30 - output ok. $1.20   
GPT-4o - input ok. $5 - output ok. $15  
o3 - input ok. $2 - output ok. $8  
o1 (zaawansowany reasoning) - input ok. $15 - output ok. $60 

* Input – tekst wysyłany do modelu
* Output – wygenerowana odpowiedź

Co do Hugging Face, znalazłem informacje, że do 0.10$ jest darmowy kredyt miesięczny, w wersji PRO i Team/Enterprise to 2$. Po wykorzystaniu płaci się pay-as-you-go, a cena zależy od czasu obliczeń, sprzętu, modelu. 